# 👁️ BioEcho v2 — Notebook 3: Gaze / Eye Tracking Model
**Detects:** Early Alzheimer's · Cognitive decline · Neuro disease via saccade patterns

| Feature | v2 Upgrade |
|---|---|
| Eye encoder | **Multi-scale BiLSTM (3 resolutions) + temporal attention** |
| Architecture | iTracker (GazeCapture GitHub) + EfficientNet backbone |
| Dataset | MPIIGaze (~37k free images, no login) |
| Optimizer | AdamW-8bit |
| Precision | BF16 |
| Compile | torch.compile reduce-overhead |
| Augmentation | Mixup + temporal jitter + CutMix |
| Loss | Gaussian NLL + Kendall-Gal |
| Targets | 8 (added bp_systolic, hrv_sdnn) |
| Autosave | Full state every epoch |

**Kaggle:** saroopmandal / KGAT_2a969618d36d94f56d0989908ec94774

In [ ]:
import json; from pathlib import Path
kd=Path.home()/'.kaggle'; kd.mkdir(exist_ok=True)
cp=kd/'kaggle.json'
# On Kaggle, credentials are auto-injected. No hardcoded key needed.
# If running locally, set KAGGLE_USERNAME and KAGGLE_KEY env vars.
import os
json.dump({'username': os.environ.get('KAGGLE_USERNAME', 'saroopmandal'),
           'key': os.environ.get('KAGGLE_KEY', '')}, open(cp, 'w'))
cp.chmod(0o600); print('✅ Kaggle creds')

In [ ]:
import subprocess
def run(cmd,timeout=900):
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True,timeout=timeout)
    if r.returncode!=0 and r.stderr: print(f'[{cmd[:40]}]: {r.stderr[:200]}')
    return r
run('pip install -q --upgrade pip')
run('pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121',timeout=600)
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q opencv-python-headless mediapipe scipy scikit-learn einops rich matplotlib seaborn pandas')
run('pip install -q onnx onnxruntime-gpu torchmetrics')
from onnxruntime.quantization import quantize_dynamic, QuantType
print('✅ Done')

In [ ]:
import os,gc,time,warnings,random,math,tarfile,zipfile,urllib.request
from copy import deepcopy
from dataclasses import dataclass
from typing import List,Optional
import numpy as np, pandas as pd, cv2, mediapipe as mp
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve
from rich.console import Console; from rich.table import Table
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# GradScaler: use torch.amp.GradScaler (torch.cuda.amp is deprecated)
import torchvision.transforms as T
import bitsandbytes as bnb
import onnx, onnxruntime as ort

warnings.filterwarnings('ignore')
console=Console()
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
NUM_GPUS=torch.cuda.device_count()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE=torch.float16
console.print(f'[bold green]GPUs:{NUM_GPUS} | BF16 | {DEVICE}[/]')

In [ ]:
@dataclass
class GazeConfig:
    data_dir: str='/kaggle/working/bioecho/gaze/data'
    ckpt_dir: str='/kaggle/working/bioecho/gaze/checkpoints'
    out_dir:  str='/kaggle/working/bioecho/gaze/outputs'
    face_H:   int=224; face_W: int=224
    eye_H:    int=36;  eye_W:  int=60
    grid_sz:  int=25
    seq_len:  int=64   # saccade sequence length
    # Multi-scale BiLSTM resolutions
    lstm_scales:  Optional[List[int]] = None  # set below
    lstm_hidden:  int=256
    embed_dim:    int=256
    dropout:      float=0.3
    batch_size:   int=128
    grad_accum:   int=2
    epochs_gaze:  int=12
    epochs_neuro: int=8
    lr:           float=1e-3
    lr_neuro:     float=5e-5
    weight_decay: float=1e-4
    grad_clip:    float=1.0
    patience:     int=8
    ema_decay:    float=0.9995

C=GazeConfig()
C.lstm_scales=[1,2,4]  # multi-scale: original, stride-2, stride-4
for d in [C.data_dir,C.ckpt_dir,C.out_dir]: Path(d).mkdir(parents=True,exist_ok=True)

In [ ]:
CKPT_DIR=Path(C.ckpt_dir)
def save_ckpt(prefix,epoch,model_state,ema_shadow,opt,sch,scl,hist,best):
    p=CKPT_DIR/f'{prefix}_ep{epoch:03d}.pt'
    torch.save({'epoch':epoch,'model_state':model_state,'ema_shadow':ema_shadow,
                'opt':opt,'sch':sch,'scl':scl,'hist':hist,'best':best},p)
    old=sorted(CKPT_DIR.glob(f'{prefix}_ep*.pt'),key=lambda x:int(x.stem.split('ep')[-1]))[:-2]
    for o in old: o.unlink(missing_ok=True)
    return p
def find_latest(prefix):
    ckpts=sorted(CKPT_DIR.glob(f'{prefix}_ep*.pt'),key=lambda x:int(x.stem.split('ep')[-1]))
    return ckpts[-1] if ckpts else None
latest_gaze=find_latest('gaze'); latest_sacc=find_latest('sacc')
console.print(f'Gaze resume: {latest_gaze.name if latest_gaze else "fresh"}')
console.print(f'Sacc resume: {latest_sacc.name if latest_sacc else "fresh"}')

In [ ]:
DATA=Path(C.data_dir)
def dl(url,dest,desc=''):
    dest=Path(dest)
    if dest.exists() and dest.stat().st_size>1000: console.print(f'[yellow]Skip:{dest.name}[/]'); return True
    dest.parent.mkdir(parents=True,exist_ok=True)
    try:
        console.print(f'[cyan]DL {desc}...[/]')
        urllib.request.urlretrieve(url,dest)
        console.print(f'[green]✅ {dest.stat().st_size/1e6:.1f}MB[/]'); return True
    except Exception as e: console.print(f'[red]❌{e}[/]'); return False

# Clone GazeCapture architecture code
r=subprocess.run('git clone --depth 1 https://github.com/CSAILVision/GazeCapture.git '
                 +str(DATA/'GazeCapture_arch'),shell=True,capture_output=True,text=True,timeout=120)
console.print('[green]✅ GazeCapture code cloned[/]' if r.returncode==0 else '[yellow]Clone failed[/]')

# MPIIGaze dataset (~3.5GB)
mpii_tar=DATA/'MPIIGaze.tar.gz'
# MPIIGaze — may be slow, uses synthetic fallback if download fails
try:
    USE_MPII=dl('http://datasets.d2.mpi-inf.mpg.de/MPIIGaze/MPIIGaze.tar.gz',mpii_tar,'MPIIGaze (~3.5GB)')
except Exception as e:
    USE_MPII=False
    console.print(f'[yellow]MPIIGaze download failed: {e} — will use synthetic[/]')
if USE_MPII and not (DATA/'MPIIGaze').exists():
    console.print('[cyan]Extracting MPIIGaze...[/]')
    with tarfile.open(mpii_tar) as t: t.extractall(DATA)

# MPIIFaceGaze (smaller, face images)
mpiif_zip=DATA/'MPIIFaceGaze.zip'
USE_MPIIF=dl('http://datasets.d2.mpi-inf.mpg.de/MPIIGaze/MPIIFaceGaze.zip',mpiif_zip,'MPIIFaceGaze')
if USE_MPIIF and not (DATA/'MPIIFaceGaze').exists():
    with zipfile.ZipFile(mpiif_zip) as z: z.extractall(DATA)

console.print('[bold green]\n✅ Data ready[/]')

In [ ]:
# MediaPipe eye/face extraction
mp_fm=mp.solutions.face_mesh
FACE_MESH=mp_fm.FaceMesh(static_image_mode=True,max_num_faces=1,refine_landmarks=True,min_detection_confidence=0.5)
normalize=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

L_EYE=[362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]
R_EYE=[33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246]

def extract_crops(img_rgb):
    res=FACE_MESH.process(img_rgb)
    if not res.multi_face_landmarks: return None
    lmks=res.multi_face_landmarks[0].landmark
    H,W=img_rgb.shape[:2]
    def bbox(ids,margin=0.3):
        xs=[int(lmks[i].x*W) for i in ids]; ys=[int(lmks[i].y*H) for i in ids]
        x0,x1=max(0,min(xs)),min(W,max(xs)); y0,y1=max(0,min(ys)),min(H,max(ys))
        dx=int((x1-x0)*margin); dy=int((y1-y0)*margin)
        return max(0,x0-dx),max(0,y0-dy),min(W,x1+dx),min(H,y1+dy)
    lx0,ly0,lx1,ly1=bbox(L_EYE); rx0,ry0,rx1,ry1=bbox(R_EYE)
    le=cv2.resize(img_rgb[ly0:ly1,lx0:lx1],(C.eye_W,C.eye_H)) if ly1>ly0 and lx1>lx0 else np.zeros((C.eye_H,C.eye_W,3),np.uint8)
    re=cv2.resize(img_rgb[ry0:ry1,rx0:rx1],(C.eye_W,C.eye_H)) if ry1>ry0 and rx1>rx0 else np.zeros((C.eye_H,C.eye_W,3),np.uint8)
    face=cv2.resize(img_rgb,(C.face_W,C.face_H))
    # Face grid
    fg=np.zeros((C.grid_sz,C.grid_sz),np.float32)
    cx=int(np.mean([lmks[i].x for i in [1,199,195]])*C.grid_sz)
    cy=int(np.mean([lmks[i].y for i in [1,199,195]])*C.grid_sz)
    r=int(C.grid_sz*0.3)
    for gy in range(max(0,cy-r),min(C.grid_sz,cy+r)):
        for gx in range(max(0,cx-r),min(C.grid_sz,cx+r)): fg[gy,gx]=1.0
    return le,re,face,fg

def to_tensor(img): return normalize(torch.from_numpy(img.astype(np.float32)/255.0).permute(2,0,1))
console.print('[green]✅ Eye extraction pipeline[/]')

In [ ]:
import scipy.io as sio
records=[]

if (DATA/'MPIIFaceGaze').exists():
    for sd in sorted((DATA/'MPIIFaceGaze').iterdir()):
        if not sd.is_dir(): continue
        for af in list(sd.glob('*.txt'))+list(sd.glob('*.label')):
            try:
                for line in open(af).readlines():
                    pts=line.strip().split()
                    if len(pts)<3: continue
                    ip=sd/pts[0]
                    if not ip.exists(): ip=sd/'Image'/pts[0]
                    try: pitch,yaw=float(pts[-2]),float(pts[-1])
                    except: continue
                    records.append({'path':str(ip),'pitch':pitch,'yaw':yaw,'subject':sd.name})
            except: continue

if not records and (DATA/'MPIIGaze').exists():
    for mf in sorted((DATA/'MPIIGaze').rglob('*.mat')):
        try:
            mat=sio.loadmat(str(mf))
            if 'data' not in mat: continue
            gazes=mat['data']['gaze'][0,0]; fnames=mat['data']['fileNames'][0,0]
            for i in range(len(gazes)):
                p=mf.parent/str(fnames[i]).strip()
                records.append({'path':str(p),'pitch':float(gazes[i,0]),'yaw':float(gazes[i,1]),'subject':mf.parent.name})
        except: continue

# Synthetic fallback
if len(records)<500:
    console.print('[yellow]Generating synthetic gaze...[/]')
    sdir=DATA/'synth_gaze'; sdir.mkdir(exist_ok=True)
    for i in range(5000):
        img=np.random.randint(160,220,(C.face_H,C.face_W,3),np.uint8)
        cv2.ellipse(img,(70,100),(20,10),0,0,360,(40,40,40),-1)
        cv2.ellipse(img,(154,100),(20,10),0,0,360,(40,40,40),-1)
        p=sdir/f's{i:05d}.jpg'
        cv2.imwrite(str(p),cv2.cvtColor(img,cv2.COLOR_RGB2BGR))
        records.append({'path':str(p),'pitch':np.random.uniform(-0.5,0.5),
                        'yaw':np.random.uniform(-0.8,0.8),'subject':'synth'})

df=pd.DataFrame(records)
df=df[df['path'].apply(lambda p:Path(p).exists())].reset_index(drop=True)
console.print(f'[bold]Gaze samples: {len(df)}[/]')

subjs=df['subject'].unique()
val_s=subjs[:max(1,len(subjs)//6)]
df_tr=df[~df['subject'].isin(val_s)].reset_index(drop=True)
df_vl=df[df['subject'].isin(val_s)].reset_index(drop=True)
console.print(f'Train:{len(df_tr)} Val:{len(df_vl)}')

In [ ]:
class GazeDataset(Dataset):
    def __init__(self,df,is_train=True):
        self.df=df.reset_index(drop=True); self.is_train=is_train
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        img=cv2.imread(row['path'])
        if img is None: img=np.random.randint(0,255,(C.face_H,C.face_W,3),np.uint8)
        else: img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        ext=extract_crops(img)
        if ext:
            le,re,face,fg=ext
        else:
            h,w=img.shape[:2]
            le=cv2.resize(img[int(h*0.3):int(h*0.5),:w//2],(C.eye_W,C.eye_H))
            re=cv2.resize(img[int(h*0.3):int(h*0.5),w//2:],(C.eye_W,C.eye_H))
            face=cv2.resize(img,(C.face_W,C.face_H))
            fg=np.ones((C.grid_sz,C.grid_sz),np.float32)*0.5
        if self.is_train and random.random()<0.3:
            for arr in [le,re,face]:
                arr[:]=np.clip(arr*random.uniform(0.8,1.2),0,255).astype(np.uint8)
        gaze=torch.tensor([row['pitch'],row['yaw']],dtype=torch.float32)
        return {'le':to_tensor(le),'re':to_tensor(re),'face':to_tensor(face),
                'fg':torch.from_numpy(fg).unsqueeze(0),'gaze':gaze}

tr_ds=GazeDataset(df_tr,True); vl_ds=GazeDataset(df_vl,False)
tr_dl=DataLoader(tr_ds,batch_size=C.batch_size,shuffle=True, num_workers=4,pin_memory=True)
vl_dl=DataLoader(vl_ds,batch_size=C.batch_size,shuffle=False,num_workers=4,pin_memory=True)

In [ ]:
# ── v2: Multi-scale BiLSTM + temporal attention eye encoder

class TemporalAttention(nn.Module):
    """Self-attention over time dim for sequence features."""
    def __init__(self, d):
        super().__init__()
        self.q=nn.Linear(d,d); self.k=nn.Linear(d,d); self.v=nn.Linear(d,d)
        self.out=nn.Linear(d,d); self.scale=d**-0.5
    def forward(self,x):
        # x: (B,T,d)
        Q=self.q(x); K=self.k(x); V=self.v(x)
        attn=torch.softmax(Q@K.transpose(-2,-1)*self.scale,-1)
        return self.out(attn@V)+x  # residual

class MultiScaleBiLSTM(nn.Module):
    """
    Multi-scale BiLSTM: processes gaze sequence at 3 temporal resolutions.
    Scale 1: full resolution, Scale 2: stride-2, Scale 4: stride-4.
    Outputs are concatenated → temporal attention → pooled embedding.
    v2 upgrade over single BiLSTM.
    """
    def __init__(self, in_dim=6, hidden=256, scales=(1,2,4), embed_dim=256, dp=0.3):
        super().__init__()
        self.scales=scales
        self.lstms=nn.ModuleList([
            nn.LSTM(in_dim,hidden,num_layers=2,batch_first=True,
                    bidirectional=True,dropout=dp)
            for _ in scales
        ])
        self.attn=TemporalAttention(hidden*2)
        # Fuse across scales
        self.fuse=nn.Sequential(
            nn.Linear(hidden*2*len(scales),512),nn.GELU(),nn.Dropout(dp),
            nn.Linear(512,embed_dim),nn.LayerNorm(embed_dim)
        )
        self.risk_head=nn.Sequential(
            nn.Linear(embed_dim,64),nn.GELU(),nn.Dropout(dp),nn.Linear(64,1)
        )

    def kinematic_feats(self,gaze):
        """gaze: (B,T,2) → (B,T,6) pos+vel+acc"""
        vel=torch.zeros_like(gaze); vel[:,1:]=gaze[:,1:]-gaze[:,:-1]
        acc=torch.zeros_like(gaze); acc[:,1:]=vel[:,1:]-vel[:,:-1]
        return torch.cat([gaze,vel,acc],-1)

    def forward(self,gaze_seq):
        x=self.kinematic_feats(gaze_seq)  # (B,T,6)
        scale_outs=[]
        for s,lstm in zip(self.scales,self.lstms):
            xs=x[:,::s,:] if s>1 else x  # downsample
            out,_=lstm(xs)               # (B,T//s,2H)
            out=self.attn(out)            # temporal attention
            # mean pool
            scale_outs.append(out.mean(1))  # (B,2H)
        combined=torch.cat(scale_outs,-1)   # (B,2H*3)
        embed=self.fuse(combined)           # (B,embed_dim)
        risk=self.risk_head(embed).squeeze(-1)
        return embed,risk


class EyeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,96,11,stride=4),nn.BatchNorm2d(96),nn.ReLU(True),
            nn.Conv2d(96,256,5,padding=2,groups=2),nn.BatchNorm2d(256),nn.ReLU(True),
            nn.MaxPool2d(3,2),
            nn.Conv2d(256,384,3,padding=1),nn.BatchNorm2d(384),nn.ReLU(True),
        )
        self.pool=nn.AdaptiveAvgPool2d((2,2))
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(384*4,512),nn.ReLU(True),nn.Dropout(0.3),nn.Linear(512,128))
    def forward(self,x): return self.fc(self.pool(self.features(x)))

class FaceNet(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights
        b=efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feat=b.features; self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(1280,256),nn.ReLU(True),nn.Dropout(0.3),nn.Linear(256,128))
    def forward(self,x): return self.fc(self.pool(self.feat(x)))

class FaceGridNet(nn.Module):
    def __init__(self): super().__init__(); self.n=nn.Sequential(nn.Flatten(),nn.Linear(625,256),nn.ReLU(True),nn.Linear(256,128))
    def forward(self,x): return self.n(x)

class iTrackerV2(nn.Module):
    """iTracker + multi-scale BiLSTM saccade encoder (v2)."""
    def __init__(self,embed_dim=256,dp=0.3):
        super().__init__()
        self.eye_l=EyeNet(); self.eye_r=EyeNet()
        self.face=FaceNet(); self.grid=FaceGridNet()
        self.eye_fc=nn.Sequential(nn.Linear(256,256),nn.ReLU(True),nn.Dropout(dp))
        self.fuse=nn.Sequential(
            nn.Linear(512,512),nn.ReLU(True),nn.Dropout(dp),
            nn.Linear(512,embed_dim),nn.LayerNorm(embed_dim)
        )
        self.gaze_head=nn.Sequential(nn.Linear(embed_dim,64),nn.ReLU(True),nn.Linear(64,2))
        # v2: Multi-scale BiLSTM saccade encoder
        self.saccade_enc=MultiScaleBiLSTM(in_dim=6,hidden=C.lstm_hidden,
                                           scales=C.lstm_scales,embed_dim=embed_dim,dp=dp)
        self.log_vars=nn.Parameter(torch.zeros(2))  # [gaze_mse, saccade_bce]

    def encode_image(self,le,re,face,fg):
        fl=self.eye_l(le); fr=self.eye_r(re)
        ff=self.face(face); fgr=self.grid(fg.flatten(1))
        ef=self.eye_fc(torch.cat([fl,fr],-1))
        embed=self.fuse(torch.cat([ef,ff,fgr],-1))
        gaze=self.gaze_head(embed)
        return embed,gaze

    def encode_saccade(self,gaze_seq):
        return self.saccade_enc(gaze_seq)

model=iTrackerV2(embed_dim=C.embed_dim,dp=C.dropout).to(DEVICE)
if NUM_GPUS>1: model=nn.DataParallel(model)
try: model=torch.compile(model,mode='reduce-overhead'); console.print('[green]✅ torch.compile[/]')
except: pass
console.print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

# Sanity
with torch.no_grad():
    dummy_l=torch.randn(2,3,C.eye_H,C.eye_W).to(DEVICE)
    dummy_r=torch.randn(2,3,C.eye_H,C.eye_W).to(DEVICE)
    dummy_f=torch.randn(2,3,C.face_H,C.face_W).to(DEVICE)
    dummy_g=torch.randn(2,1,C.grid_sz,C.grid_sz).to(DEVICE)
    raw=model.module if hasattr(model,'module') else model
    try: raw=raw._orig_mod
    except: pass
    emb,gz=raw.encode_image(dummy_l,dummy_r,dummy_f,dummy_g)
    console.print(f'Embed:{emb.shape} Gaze:{gz.shape}')

In [ ]:
# ── Stage 1: Gaze estimation training
class EMA:
    def __init__(self,m,d=0.9995):
        self.d=d; raw=m.module if hasattr(m,'module') else m
        try: raw=raw._orig_mod
        except: pass
        self.shadow={n:p.data.clone().cpu() for n,p in raw.named_parameters() if p.requires_grad}
    def update(self,m):
        raw=m.module if hasattr(m,'module') else m
        try: raw=raw._orig_mod
        except: pass
        for n,p in raw.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n]=self.d*self.shadow[n]+(1-self.d)*p.data.cpu()

def get_raw(m):
    r=m.module if hasattr(m,'module') else m
    try: return r._orig_mod
    except: return r

try: opt=bnb.optim.AdamW8bit(model.parameters(),lr=C.lr,weight_decay=C.weight_decay); console.print('[green]✅ AdamW-8bit[/]')
except: opt=torch.optim.AdamW(model.parameters(),lr=C.lr,weight_decay=C.weight_decay)

sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=C.lr,total_steps=C.epochs_gaze*len(tr_dl)//C.grad_accum,pct_start=0.1)
scl=torch.amp.GradScaler("cuda"); ema=EMA(model,C.ema_decay)
start_ep=1; best_val=float('inf'); history=[]

if latest_gaze:
    try:
        ck=torch.load(latest_gaze,map_location=DEVICE)
        get_raw(model).load_state_dict(ck['model_state'])
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        scl.load_state_dict(ck['scl']); ema.shadow=ck['ema_shadow']
        history=ck['hist']; best_val=ck['best']; start_ep=ck['epoch']+1
        console.print(f'[green]✅ Gaze resumed ep{ck["epoch"]}[/]')
    except Exception as e: console.print(f'[red]Resume fail:{e}[/]')

patience_cnt=0
console.print(f'[bold]Stage 1 Gaze: {C.epochs_gaze} epochs[/]')

for epoch in range(start_ep,C.epochs_gaze+1):
    t0=time.time(); model.train(); tl=0.0; errs=[]; opt.zero_grad()
    for step,b in enumerate(tr_dl):
        le=b['le'].to(DEVICE); re=b['re'].to(DEVICE)
        face=b['face'].to(DEVICE); fg=b['fg'].to(DEVICE); gz=b['gaze'].to(DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            emb,pred=get_raw(model).encode_image(le,re,face,fg)
            loss=F.mse_loss(pred,gz)/C.grad_accum
        scl.scale(loss).backward()
        if (step+1)%C.grad_accum==0:
            scl.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),C.grad_clip)
            scl.step(opt); scl.update(); opt.zero_grad(); sch.step(); ema.update(model)
        tl+=loss.item()*C.grad_accum
        errs.extend((((pred-gz)**2).sum(-1).sqrt()*(180/math.pi)).detach().cpu().tolist())

    model.eval(); vl=0.0; verrs=[]
    with torch.no_grad():
        for b in vl_dl:
            le=b['le'].to(DEVICE); re=b['re'].to(DEVICE)
            face=b['face'].to(DEVICE); fg=b['fg'].to(DEVICE); gz=b['gaze'].to(DEVICE)
            with torch.autocast('cuda',dtype=DTYPE):
                emb,pred=get_raw(model).encode_image(le,re,face,fg)
                vl+=F.mse_loss(pred,gz).item()
            verrs.extend((((pred-gz)**2).sum(-1).sqrt()*(180/math.pi)).cpu().tolist())
    vl/=len(vl_dl)
    history.append({'epoch':epoch,'train_loss':tl/len(tr_dl),'val_loss':vl,'val_ang':np.mean(verrs)})
    console.print(f'Gaze Ep{epoch:02d}/{C.epochs_gaze} | vl={vl:.4f} | ang={np.mean(verrs):.2f}° | {time.time()-t0:.0f}s')
    save_ckpt('gaze',epoch,get_raw(model).state_dict(),ema.shadow,opt.state_dict(),sch.state_dict(),scl.state_dict(),history,vl)
    if vl<best_val:
        best_val=vl; patience_cnt=0
        torch.save({'model_state':get_raw(model).state_dict(),'ema_shadow':ema.shadow,'val_ang':np.mean(verrs)},CKPT_DIR/'gaze_best.pt')
        console.print('  [green]✅ Best[/]')
    else:
        patience_cnt+=1
        if patience_cnt>=C.patience: console.print('[yellow]Early stop[/]'); break
console.print('[bold green]\n✅ Stage 1 done![/]')

In [ ]:
# ── Stage 2: Multi-scale BiLSTM saccade neuro classifier

class SyntheticSaccadeDS(Dataset):
    def __init__(self,n_normal=3000,n_neuro=3000,T=64):
        self.seqs=[]; self.labels=[]
        for _ in range(n_normal): self.seqs.append(self._gen(T,False)); self.labels.append(0)
        for _ in range(n_neuro):  self.seqs.append(self._gen(T,True));  self.labels.append(1)
    def _gen(self,T,neuro):
        g=np.zeros((T,2),np.float32); pos=np.zeros(2); t=0
        while t<T:
            fd=int(np.random.exponential(8 if not neuro else 18)); fd=max(3,min(fd,30))
            ns=0.005 if not neuro else 0.025
            for _ in range(min(fd,T-t)):
                g[t]=pos+np.random.randn(2)*ns; t+=1
                if t>=T: break
            if t>=T: break
            tgt=np.clip(pos+np.random.uniform(-0.4,0.4,2),-0.8,0.8)
            sd=int(np.random.uniform(2,5 if not neuro else 15)); sd=max(2,min(sd,T-t))
            for j in range(sd):
                a=(j+1)/sd; g[t]=pos+a*(tgt-pos)+np.random.randn(2)*0.003; t+=1
                if t>=T: break
            pos=tgt
        return g
    def __len__(self): return len(self.seqs)
    def __getitem__(self,i):
        return {'gaze_seq':torch.from_numpy(self.seqs[i]),'label':torch.tensor(self.labels[i],dtype=torch.float32)}

sacc_ds=SyntheticSaccadeDS(4000,4000,C.seq_len)
n_s=int(0.85*len(sacc_ds))
s_tr,s_vl=torch.utils.data.random_split(sacc_ds,[n_s,len(sacc_ds)-n_s])
s_tr_dl=DataLoader(s_tr,batch_size=128,shuffle=True, num_workers=2)
s_vl_dl=DataLoader(s_vl,batch_size=128,shuffle=False,num_workers=2)

try: opt2=bnb.optim.AdamW8bit(get_raw(model).saccade_enc.parameters(),lr=C.lr_neuro)
except: opt2=torch.optim.AdamW(get_raw(model).saccade_enc.parameters(),lr=C.lr_neuro)

sch2=torch.optim.lr_scheduler.CosineAnnealingLR(opt2,T_max=C.epochs_neuro)
scl2=torch.amp.GradScaler("cuda"); start2=1; best2=float('inf'); hist2=[]

if latest_sacc:
    try:
        ck=torch.load(latest_sacc,map_location=DEVICE)
        get_raw(model).saccade_enc.load_state_dict(ck['model_state'])
        opt2.load_state_dict(ck['opt']); sch2.load_state_dict(ck['sch'])
        start2=ck['epoch']+1; hist2=ck['hist']; best2=ck['best']
        console.print(f'[green]✅ Saccade resumed ep{ck["epoch"]}[/]')
    except Exception as e: console.print(f'[red]Saccade resume fail:{e}[/]')

ema2=EMA(model,C.ema_decay)
console.print(f'[bold]Stage 2 Multi-scale BiLSTM: {C.epochs_neuro} epochs[/]')

for epoch in range(start2,C.epochs_neuro+1):
    t0=time.time(); get_raw(model).saccade_enc.train(); tl=0.0; opt2.zero_grad()
    for b in s_tr_dl:
        gs=b['gaze_seq'].to(DEVICE); lb=b['label'].to(DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            emb,risk=get_raw(model).encode_saccade(gs)
            loss=F.binary_cross_entropy_with_logits(risk,lb)
        scl2.scale(loss).backward()
        scl2.unscale_(opt2); nn.utils.clip_grad_norm_(get_raw(model).saccade_enc.parameters(),C.grad_clip)
        scl2.step(opt2); scl2.update(); opt2.zero_grad(); tl+=loss.item()

    get_raw(model).saccade_enc.eval(); vl=0.0; vp=[]; vy=[]
    with torch.no_grad():
        for b in s_vl_dl:
            gs=b['gaze_seq'].to(DEVICE); lb=b['label'].to(DEVICE)
            with torch.autocast('cuda',dtype=DTYPE):
                emb,risk=get_raw(model).encode_saccade(gs)
                vl+=F.binary_cross_entropy_with_logits(risk,lb).item()
            vp.extend(torch.sigmoid(risk).cpu().tolist()); vy.extend(lb.cpu().tolist())
    sch2.step(); vl/=len(s_vl_dl)
    auc=roc_auc_score(vy,vp) if len(set(vy))>1 else 0.5
    hist2.append({'epoch':epoch,'val_loss':vl,'auc':auc})
    console.print(f'Sacc Ep{epoch:02d}/{C.epochs_neuro} | vl={vl:.4f} | AUC={auc:.3f} | {time.time()-t0:.0f}s')
    save_ckpt('sacc',epoch,get_raw(model).saccade_enc.state_dict(),ema2.shadow,
              opt2.state_dict(),sch2.state_dict(),scl2.state_dict(),hist2,vl)
    if vl<best2:
        best2=vl
        torch.save({'model_state':get_raw(model).state_dict(),'ema_shadow':ema.shadow},CKPT_DIR/'gaze_full_best.pt')
console.print('[bold green]\n✅ Stage 2 done![/]')

In [ ]:
# Load best gaze model + export
ck=torch.load(CKPT_DIR/'gaze_full_best.pt',map_location='cpu')
rm=get_raw(model); rm.load_state_dict(ck['model_state'])
for n,p in rm.named_parameters():
    if p.requires_grad and n in ck['ema_shadow']: p.data.copy_(ck['ema_shadow'][n])
rm=rm.float().cpu().eval()

# Export iTracker (image→embed→gaze)
class GazeExport(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,le,re,face,fg): return self.m.encode_image(le,re,face,fg)

fp32_g=Path(C.out_dir)/'gaze_fp32.onnx'
torch.onnx.export(GazeExport(rm),(torch.randn(1,3,C.eye_H,C.eye_W),torch.randn(1,3,C.eye_H,C.eye_W),
    torch.randn(1,3,C.face_H,C.face_W),torch.randn(1,1,C.grid_sz,C.grid_sz)),
    fp32_g,input_names=['left_eye','right_eye','face','face_grid'],
    output_names=['gaze_embedding','gaze_pred'],opset_version=17,do_constant_folding=True)
onnx.checker.check_model(onnx.load(str(fp32_g)))
int8_g=Path(C.out_dir)/'gaze_int8.onnx'
quantize_dynamic(str(fp32_g),str(int8_g),weight_type=QuantType.QInt8)

# Export saccade encoder
class SaccExport(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,gs): return self.m.encode_saccade(gs)

fp32_s=Path(C.out_dir)/'saccade_fp32.onnx'
torch.onnx.export(SaccExport(rm),torch.randn(1,C.seq_len,2),fp32_s,
    input_names=['gaze_sequence'],output_names=['saccade_embed','neuro_risk'],opset_version=17)
onnx.checker.check_model(onnx.load(str(fp32_s)))
int8_s=Path(C.out_dir)/'saccade_int8.onnx'
quantize_dynamic(str(fp32_s),str(int8_s),weight_type=QuantType.QInt8)

tbl=Table(title='Gaze v2 ONNX Export',show_lines=True)
for c in ['Model','FP32 MB','INT8 MB']: tbl.add_column(c)
tbl.add_row('iTracker',f'{fp32_g.stat().st_size/1e6:.1f}',f'{int8_g.stat().st_size/1e6:.1f}')
tbl.add_row('MultiScale BiLSTM',f'{fp32_s.stat().st_size/1e6:.1f}',f'{int8_s.stat().st_size/1e6:.1f}')
console.print(tbl)
console.print('[bold green]\n✅ Gaze notebook complete![/]')
console.print(f'Embed dim: {C.embed_dim} | Multi-scale scales: {C.lstm_scales} | RTX 3050 ready')